# Pilot: fine-tuning ADM as a one-step MMSE denoiser (diffusion training restricted to the denoising steps)

`diffusion-tweedie-benchmark` showed that the frozen ADM 256×256 unconditional model, used as a one-step posterior
mean at the step matched to σ, reaches DMID-d's published state of the art (1000 evaluations) with one or two
evaluations, and that calibration, symmetry ensembles and blending add at most ~0.02 dB. This pilot asks whether a
short fine-tune moves the ceiling.

For y = x + σn, x_t = sqrt(ā)(2y − 1) with ā = 1/(1 + 4σ²) is exactly the forward diffusion state at the matched
continuous step with ε = n. So ADM's own ε loss on such states, with σ ~ U[5, 60]/255, trains the one-step
estimate directly. Training data: DIV2K train (800 images), random crops at a random scale in [¼, 1], horizontal
flips; validation on DIV2K valid 0801–0810, never on a test set. torchrun DDP on both T4s, fp16 autocast, Muon (AdamW
for the first and last convolutions, biases and norms), EMA on the CPU.

Evaluation: base and fine-tuned ADM, one step each (the flip ensemble adds ~0.013 dB to either), on CBSD68, Kodak24, McMaster
(σ ∈ {15, 25, 50}, DMID's protocol, the same seeds as `diffusion-tweedie-benchmark`) and the first
`N_URBAN` Urban100 images.

Kaggle: GPU T4 ×2, internet on, no attached inputs.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lpips"], check=True)

In [ ]:
import glob, io, json, math, os, queue, re, subprocess, sys, threading, time, traceback, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from PIL import Image
from skimage.restoration import estimate_sigma

TEST_MODE = os.environ.get("TEST_MODE") == "1"     # CPU smoke test: tiny random ADM, one process, tiny crops
TRAIN_MINUTES = 0.4 if TEST_MODE else 480           # pilot (kernel adm-tweedie-finetune): 100
SIGMAS = [15, 25, 50]
DATASETS = ["McMaster", "Kodak24", "CBSD68", "Urban100"]   # seeds as diffusion-tweedie-benchmark (first three)
N_URBAN = 2 if TEST_MODE else 100                   # pilot: 30
EVAL_FLIP = True                                    # also score the fine-tuned model with the horizontal-flip average (pilot: off)
RESUME = "adm_ft_ema.pt"                            # continue from this file if an earlier run's output is attached (pilot: none)
TIME_LIMIT_H = 11.5
DEVICES = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())] or [torch.device("cpu")]
assert TEST_MODE or DEVICES[0].type == "cuda", "GPU required"
torch.backends.cudnn.benchmark = False
CACHE, OUT = os.environ.get("CACHE_DIR", "/tmp/cache"), "outputs"
for d in (CACHE, OUT):
    os.makedirs(d, exist_ok=True)
T_START, TIMES = time.time(), {}
print([str(d) for d in DEVICES], torch.__version__, "TEST_MODE" if TEST_MODE else "")


def download(url, path, size=None):
    """Resumable download with progress; no-op when the file is already complete."""
    have = os.path.getsize(path) if os.path.exists(path) else 0
    if have and (size is None or have == size):
        return path
    if size and have > size:
        have = 0
    req = urllib.request.Request(url, headers={"Range": f"bytes={have}-"} if have else {})
    with urllib.request.urlopen(req, timeout=60) as r:
        if have and r.status != 206:
            have = 0  # server ignored the Range header: start over
        total, t0, got, mark = have + int(r.headers.get("Content-Length", 0)), time.time(), 0, 0
        with open(path, "ab" if have else "wb") as f:
            while chunk := r.read(1 << 23):
                f.write(chunk)
                got += len(chunk)
                if got - mark >= 1 << 30:
                    mark = got
                    print(f"  {path.split('/')[-1]}: {(have + got) / 1e9:.1f}/{total / 1e9:.1f} GB  {got / 1e6 / (time.time() - t0):.0f} MB/s", flush=True)
    if size:
        assert os.path.getsize(path) == size, f"incomplete download: {path}"
    return path


def natkey(s):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]


t0 = time.time()
FILES = {
    "adm": ("https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion_uncond.pt", 2_211_383_297),
    "datasets": ("https://github.com/Li-Tong-621/DMID/releases/download/v1.0/Datasets.zip", 367_018_356),
    "div2k_valid": ("https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip", 448_993_893),
    "div2k_train": ("https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip", 3_530_603_713),
}
if TEST_MODE:
    FILES = {k: v for k, v in FILES.items() if k in ("datasets", "div2k_valid")}
with ThreadPoolExecutor(4) as ex:
    P = dict(zip(FILES, ex.map(lambda kv: download(kv[1][0], f"{CACHE}/{kv[0]}{os.path.splitext(kv[1][0])[1]}", kv[1][1]), FILES.items())))
GD = f"{CACHE}/guided-diffusion"
if not os.path.isdir(GD):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/openai/guided-diffusion", GD], check=True)
TIMES["downloads"] = time.time() - t0
print(f"downloads {TIMES['downloads'] / 60:.1f} min")

## 1 · Training data (DIV2K train → one memory-mapped uint8 file) and the validation set (DIV2K valid 0801–0810)

In [ ]:
t0 = time.time()
DATA_FILE, DATA_INDEX, VAL_FILE = f"{CACHE}/div2k_train.u8", f"{CACHE}/div2k_train.json", f"{CACHE}/val.pt"
train_zip = P["div2k_valid"] if TEST_MODE else P["div2k_train"]
with zipfile.ZipFile(train_zip) as z:
    names = sorted(n for n in z.namelist() if n.lower().endswith(".png"))[: 4 if TEST_MODE else None]


def decode(name):
    with zipfile.ZipFile(train_zip) as z:
        im = np.asarray(Image.open(io.BytesIO(z.read(name))).convert("RGB"))
    return im[:300, :400].copy() if TEST_MODE else im


index, offset = [], 0
with open(DATA_FILE, "wb") as f, ThreadPoolExecutor(4) as ex:
    for name, im in zip(names, ex.map(decode, names)):
        f.write(im.tobytes())
        index.append(dict(name=name.split("/")[-1], h=im.shape[0], w=im.shape[1], offset=offset))
        offset += im.size
json.dump(index, open(DATA_INDEX, "w"))
print(f"{len(index)} training images, {offset / 1e9:.2f} GB")


def noisy_of(clean_u8, sigma, seed):
    """DMID's protocol: Gaussian noise normalised to sample std exactly sigma/255, added in float, never clipped."""
    x = torch.from_numpy(clean_u8).permute(2, 0, 1).float() / 255
    n = torch.randn(x.shape, generator=torch.Generator().manual_seed(seed))
    return x + (sigma / 255) * n / n.std(unbiased=False)


VAL = []
with zipfile.ZipFile(P["div2k_valid"]) as z:
    for k, n in enumerate(sorted(n for n in z.namelist() if n.lower().endswith(".png"))[: 2 if TEST_MODE else 10]):
        im = Image.open(io.BytesIO(z.read(n))).convert("RGB")
        half = TF.center_crop(im.resize((im.width // 2, im.height // 2), Image.LANCZOS), [64, 64] if TEST_MODE else [512, 512])
        clean = np.asarray(half).copy()
        for s in SIGMAS:
            VAL.append((torch.from_numpy(clean).permute(2, 0, 1).contiguous(), s, noisy_of(clean, s, 9_000_000 + 1000 * s + k)))
torch.save(VAL, VAL_FILE)
TIMES["data"] = time.time() - t0
print(f"data {TIMES['data'] / 60:.1f} min")

## 2 · Fine-tuning (torchrun, one process per GPU)

In [ ]:
TRAIN_SRC = r'''"""Fine-tune ADM 256x256 uncond as a one-step MMSE denoiser: diffusion training restricted to the denoising steps.

For y = x + σn (unclipped), x_t = sqrt(ā)(2y - 1) with ā = 1 / (1 + 4σ²) is exactly the forward diffusion state at
the matching (continuous) step with ε = n, so the standard ε loss on these states trains the Tweedie estimate
x̂ = (x_t - sqrt(1 - ā) ε̂) / sqrt(ā) directly. Launched with torchrun; one process per GPU (gloo on CPU for tests).
"""
import json
from contextlib import nullcontext
import math
import os
import sys
import time

import numpy as np
import torch
import torch.distributed as dist
import torch.nn.functional as F
import torch.utils.checkpoint

CFG = json.loads(os.environ["FT_CONFIG"])
TEST = CFG.get("test_mode", False)
if TEST:  # CPU smoke test without torchrun (its TCP store needs libuv on Windows)
    dist.init_process_group("gloo", init_method="file:///" + os.path.abspath(CFG["data_file"] + ".store").replace("\\", "/"), rank=0, world_size=1)
else:
    dist.init_process_group("nccl")
RANK, WORLD = dist.get_rank(), dist.get_world_size()
DEV = torch.device("cpu") if TEST else torch.device(f"cuda:{RANK}")
if not TEST:
    torch.cuda.set_device(DEV)
torch.manual_seed(1234 + RANK)
RNG = np.random.default_rng(1234 + RANK)

sys.path[:0] = [CFG["guided_diffusion"]]
from guided_diffusion import unet as gd_unet  # noqa: E402
from guided_diffusion.script_util import create_model  # noqa: E402
from muon import Muon, muon_parameter_groups  # noqa: E402  laboratory/optimization/muon.py, written next to this script


def checkpoint(func, inputs, params, flag):
    """torch.utils.checkpoint instead of ADM's own (which recomputes outside autocast)."""
    if flag and torch.is_grad_enabled():
        return torch.utils.checkpoint.checkpoint(func, *inputs, use_reentrant=False)
    return func(*inputs)


def sdpa_attention(self, qkv):
    bs, width, length = qkv.shape
    ch = width // (3 * self.n_heads)
    q, k, v = qkv.reshape(bs * self.n_heads, ch * 3, length).split(ch, dim=1)
    out = F.scaled_dot_product_attention(q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2))
    return out.transpose(1, 2).reshape(bs, -1, length)


gd_unet.checkpoint = checkpoint
gd_unet.QKVAttentionLegacy.forward = sdpa_attention

model = create_model(**CFG["adm_flags"], use_fp16=False, use_checkpoint=True)
if CFG.get("adm_checkpoint"):
    model.load_state_dict(torch.load(CFG["adm_checkpoint"], map_location="cpu", weights_only=True))
if CFG.get("init_weights"):  # continue from an earlier fine-tune (fp16 EMA weights)
    model.load_state_dict({k: v.float() for k, v in torch.load(CFG["init_weights"], map_location="cpu", weights_only=True).items()})
model.to(DEV).train()
ddp = torch.nn.parallel.DistributedDataParallel(model, device_ids=None if TEST else [RANK], gradient_as_bucket_view=True)
opt = Muon(muon_parameter_groups(model, adamw_modules=("input_blocks.0.0", "out.2")), lr=CFG["lr"], weight_decay=0.0)  # first/last conv on AdamW
scaler = torch.amp.GradScaler("cuda", enabled=not TEST)

BETAS = torch.linspace(1e-4, 0.02, 1000, dtype=torch.float64)
NEG_LOG_ABAR = (-torch.log(torch.cumprod(1 - BETAS, 0))).numpy()
INDEX = json.load(open(CFG["data_index"]))
DATA = np.memmap(CFG["data_file"], dtype=np.uint8, mode="r")
CROP = CFG["crop"]


def batch(n):
    """Random crops (scale U[s_min, 1], antialiased to CROP px, re-quantised), horizontal flips, σ ~ U[σ_min, σ_max]."""
    xs = []
    for _ in range(n):
        im = INDEX[RNG.integers(len(INDEX))]
        h, w, off = im["h"], im["w"], im["offset"]
        s = RNG.uniform(CFG["scale_min"], 1.0)
        c = min(int(round(CROP / s)), h, w)
        i, j = RNG.integers(h - c + 1), RNG.integers(w - c + 1)
        img = DATA[off: off + h * w * 3].reshape(h, w, 3)[i:i + c, j:j + c]
        x = torch.from_numpy(np.ascontiguousarray(img)).to(DEV).permute(2, 0, 1)[None].float()
        if c != CROP:
            x = F.interpolate(x, size=(CROP, CROP), mode="bicubic", antialias=True, align_corners=False).clamp(0, 255).round()
        if RNG.random() < 0.5:
            x = x.flip(-1)
        xs.append(x / 255)
    x = torch.cat(xs)
    sigma = torch.from_numpy(RNG.uniform(CFG["sigma_min"], CFG["sigma_max"], n) / 255).float().to(DEV)
    t = torch.from_numpy(np.interp(np.log1p(4 * sigma.double().cpu().numpy() ** 2), NEG_LOG_ABAR, np.arange(1000))).float().to(DEV)
    ab = (1 / (1 + 4 * sigma ** 2)).view(-1, 1, 1, 1)
    n_ = torch.randn_like(x)
    xt = ab.sqrt() * (2 * (x + sigma.view(-1, 1, 1, 1) * n_) - 1)
    return xt, t, n_


VAL = torch.load(CFG["val_file"]) if RANK == 0 else None   # list of (clean uint8 [3,H,W], sigma, noisy float [3,H,W])


@torch.no_grad()
def validate():
    model.eval()
    psnr = {}
    for clean, sigma, noisy in VAL:
        y = noisy[None].to(DEV)
        s01 = sigma / 255
        ab = 1 / (1 + 4 * s01 ** 2)
        t = float(np.interp(math.log1p(4 * s01 ** 2), NEG_LOG_ABAR, np.arange(1000)))
        h, w = y.shape[-2:]
        yp = F.pad(y, (0, (-w) % 64, 0, (-h) % 64), mode="reflect")
        xt = math.sqrt(ab) * (2 * yp - 1)
        with torch.autocast("cuda", dtype=torch.float16, enabled=not TEST):
            eps = model(xt, torch.full((1,), t, device=DEV))[:, :3].float()
        x0 = ((xt - math.sqrt(1 - ab) * eps) / math.sqrt(ab) + 1) / 2
        out = (x0[0, :, :h, :w].clamp(0, 1) * 255).round()
        mse = (out - clean.to(DEV).float()).pow(2).mean().item()
        psnr.setdefault(sigma, []).append(10 * math.log10(255 ** 2 / mse))
    model.train()
    return {s: float(np.mean(v)) for s, v in psnr.items()}


ema = {k: v.detach().float().cpu().clone() for k, v in model.state_dict().items()} if RANK == 0 else None
decay = 0.5 ** (CFG["ema_every"] / CFG["ema_half_life"])


def ema_update():
    with torch.no_grad():
        for k, v in model.state_dict().items():
            ema[k].mul_(decay).add_(v.detach().float().cpu(), alpha=1 - decay)


def swap_in(weights):
    saved = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(weights)
    return saved

log, total_steps, step = [], None, 0
if RANK == 0:
    swap = swap_in(ema)
    log.append(dict(step=0, minutes=0.0, **{f"val_psnr_{s}": p for s, p in validate().items()}))
    model.load_state_dict(swap)
    print("step 0 validation", log[-1], flush=True)
t_start = time.time()
while total_steps is None or step < total_steps:
    if step == CFG["measure_steps"] and total_steps is None:
        per_step = torch.tensor([(time.time() - t_start) / step], device=DEV)
        dist.broadcast(per_step, 0)
        total_steps = max(step + 1, int(CFG["train_minutes"] * 60 / per_step.item()))
        if RANK == 0:
            print(f"{per_step.item():.2f} s/step -> {total_steps} steps", flush=True)
    warm = min(1.0, (step + 1) / CFG["warmup"])
    cosine = 0.5 * (1 + math.cos(math.pi * step / total_steps)) if total_steps else 1.0
    for g in opt.param_groups:
        g["lr"] = CFG["lr"] * warm * (CFG["lr_floor"] + (1 - CFG["lr_floor"]) * cosine)
    opt.zero_grad(set_to_none=True)
    for a in range(CFG["accum"]):
        xt, t, target = batch(CFG["micro_batch"])
        with (ddp.no_sync() if a < CFG["accum"] - 1 else nullcontext()):
            with torch.autocast("cuda", dtype=torch.float16, enabled=not TEST):
                loss = F.mse_loss(ddp(xt, t)[:, :3].float(), target) / CFG["accum"]
            scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt)
    scaler.update()
    loss = loss * CFG["accum"]
    if step == 0 and RANK == 0 and not TEST:
        print(f"peak GPU memory after the first step: {torch.cuda.max_memory_allocated(DEV) / 2**30:.2f} GiB", flush=True)
    step += 1
    if RANK == 0 and step % CFG["ema_every"] == 0:
        ema_update()
    if RANK == 0 and step % CFG["log_every"] == 0:
        print(f"step {step} loss {loss.item():.4f} lr {opt.param_groups[0]['lr']:.2e} {(time.time() - t_start) / 60:.1f} min", flush=True)
    if total_steps and (step % CFG["val_every"] == 0 or step == total_steps):
        if RANK == 0:
            swap = swap_in(ema)
            log.append(dict(step=step, minutes=(time.time() - t_start) / 60, loss=loss.item(), **{f"val_psnr_{s}": p for s, p in validate().items()}))
            model.load_state_dict(swap)
            print("validation", log[-1], flush=True)
        dist.barrier()
if RANK == 0:
    torch.save({k: v.half() for k, v in ema.items()}, CFG["out_weights"])
    json.dump(log, open(CFG["out_log"], "w"), indent=1)
    print(f"saved {CFG['out_weights']} after {step} steps, {(time.time() - t_start) / 60:.1f} min", flush=True)
dist.barrier()
dist.destroy_process_group()
'''
MUON_SRC = r'''"""Muon: momentum orthogonalised by Newton–Schulz, for hidden weight matrices.

Muon (Jordan et al., 2024) replaces the update of each weight matrix with the
nearest semi-orthogonal matrix to its Nesterov momentum, computed by five
quintic Newton–Schulz iterations; convolution kernels are flattened to
``(out, in · kh · kw)``. It is meant for hidden matrices only, so embeddings,
the output layer, biases and normalisation gains are updated by AdamW inside
the same optimizer, as in the reference implementation.

Muon updates are scaled by ``0.2 · sqrt(max(rows, columns))`` (Liu et al.,
2025, "Muon is Scalable for LLM Training"), which matches their RMS to AdamW's:
learning rates and weight decay tuned for AdamW carry over unchanged.
"""

from __future__ import annotations

import math
from typing import Iterable

import torch
import torch.nn as nn

NEWTON_SCHULZ = (3.4445, -4.7750, 2.0315)


def newton_schulz(matrix: torch.Tensor, steps: int = 5) -> torch.Tensor:
    """Approximate the orthogonal factor ``U V^T`` of ``matrix = U S V^T``; singular values end up in about [0.7, 1.2]."""
    a, b, c = NEWTON_SCHULZ
    native_bf16 = matrix.is_cuda and torch.cuda.get_device_capability(matrix.device)[0] >= 8
    x = matrix.to(torch.bfloat16 if native_bf16 else torch.float32)
    tall = x.size(-2) > x.size(-1)
    if tall:
        x = x.mT
    x = x / (x.norm(dim=(-2, -1), keepdim=True) + 1e-7)
    for _ in range(steps):
        gram = x @ x.mT
        x = a * x + (b * gram + c * gram @ gram) @ x
    return (x.mT if tall else x).to(matrix.dtype)


def muon_parameter_groups(model: nn.Module, adamw_modules: Iterable[str] = ("head", "lm_head")) -> list[dict]:
    """Split ``model``'s parameters into a Muon group and an AdamW group.

    AdamW gets every parameter with fewer than two dimensions, every
    ``nn.Embedding`` weight (with any output layer tied to it) and every
    parameter of a module named in ``adamw_modules``, matched against the
    qualified name or its last component. Muon gets the remaining matrices.
    """
    names = set(adamw_modules)
    adamw = {
        id(p)
        for name, module in model.named_modules()
        if isinstance(module, nn.Embedding) or name in names or name.rsplit(".", 1)[-1] in names
        for p in module.parameters(recurse=False)
    }
    muon, other = [], []
    for p in model.parameters():  # tied weights appear once
        (other if p.ndim < 2 or id(p) in adamw else muon).append(p)
    return [{"params": muon, "use_muon": True}, {"params": other, "use_muon": False}]


class Muon(torch.optim.Optimizer):
    """Muon for hidden weight matrices, AdamW for every other parameter.

    ``params`` holds tensors, split by shape (two or more dimensions go to
    Muon), or groups with a ``use_muon`` flag such as
    :func:`muon_parameter_groups` returns. ``lr`` and decoupled
    ``weight_decay`` apply to both kinds; ``momentum``, ``nesterov`` and
    ``ns_steps`` to Muon; ``betas`` and ``eps`` to AdamW.
    """

    def __init__(
        self,
        params,
        lr: float = 3e-4,
        weight_decay: float = 0.0,
        momentum: float = 0.95,
        nesterov: bool = True,
        ns_steps: int = 5,
        betas: tuple[float, float] = (0.9, 0.999),
        eps: float = 1e-8,
    ):
        params = list(params)
        if params and not isinstance(params[0], dict):
            params = [
                {"params": [p for p in params if p.ndim >= 2], "use_muon": True},
                {"params": [p for p in params if p.ndim < 2], "use_muon": False},
            ]
        groups = [{**group, "params": list(group["params"])} for group in params]
        if any("use_muon" not in group for group in groups):
            raise ValueError("every parameter group needs a 'use_muon' flag")
        defaults = dict(
            lr=lr, weight_decay=weight_decay, momentum=momentum, nesterov=nesterov,
            ns_steps=ns_steps, betas=betas, eps=eps,
        )
        super().__init__([group for group in groups if group["params"]], defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            update = self._muon_update if group["use_muon"] else self._adamw_update
            for p in group["params"]:
                if p.grad is None:
                    continue
                direction = update(p.grad, self.state[p], group)
                p.mul_(1 - group["lr"] * group["weight_decay"])
                p.add_(direction, alpha=-group["lr"])
        return loss

    @staticmethod
    def _muon_update(grad, state, group):
        if not state:
            state["momentum_buffer"] = torch.zeros_like(grad)
        buffer = state["momentum_buffer"]
        buffer.lerp_(grad, 1 - group["momentum"])
        momentum = grad.lerp(buffer, group["momentum"]) if group["nesterov"] else buffer
        matrix = newton_schulz(momentum.reshape(len(momentum), -1), group["ns_steps"])
        return matrix.reshape_as(grad) * (0.2 * math.sqrt(max(matrix.shape)))

    @staticmethod
    def _adamw_update(grad, state, group):
        beta1, beta2 = group["betas"]
        if not state:
            state["step"] = 0
            state["exp_avg"] = torch.zeros_like(grad)
            state["exp_avg_sq"] = torch.zeros_like(grad)
        state["step"] += 1
        state["exp_avg"].lerp_(grad, 1 - beta1)
        state["exp_avg_sq"].mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
        corrected = (state["exp_avg_sq"] / (1 - beta2 ** state["step"])).sqrt().add_(group["eps"])
        return state["exp_avg"] / (1 - beta1 ** state["step"]) / corrected
'''  # laboratory/optimization/muon.py
TRAIN_SCRIPT = f"{CACHE}/train_ft.py"
open(TRAIN_SCRIPT, "w").write(TRAIN_SRC)
open(f"{CACHE}/muon.py", "w").write(MUON_SRC)
ADM_FLAGS = dict(image_size=256, num_channels=256, num_res_blocks=2, learn_sigma=True, class_cond=False,
                 attention_resolutions="32,16,8", num_head_channels=64, resblock_updown=True, use_scale_shift_norm=True)
if TEST_MODE:
    ADM_FLAGS.update(num_channels=32, num_head_channels=16, num_res_blocks=1)
INPUT_ROOTS = ["/kaggle/input", os.environ.get("INPUT_DIR", "/nonexistent")]
INIT = next((p for root in INPUT_ROOTS for p in sorted(glob.glob(f"{root}/**/{RESUME}", recursive=True))), None)
print("continuing from", INIT or "the pretrained ADM")
CONFIG = dict(test_mode=TEST_MODE, guided_diffusion=GD, adm_flags=ADM_FLAGS, adm_checkpoint=None if TEST_MODE else P["adm"], init_weights=INIT,
              data_file=DATA_FILE, data_index=DATA_INDEX, val_file=VAL_FILE, crop=64 if TEST_MODE else 256,
              scale_min=0.25, sigma_min=5.0, sigma_max=60.0, micro_batch=2, accum=2, lr=2e-5, lr_floor=0.1, warmup=2 if TEST_MODE else 100,
              ema_every=2 if TEST_MODE else 10, ema_half_life=4 if TEST_MODE else 150, log_every=2 if TEST_MODE else 50,
              val_every=4 if TEST_MODE else 400, measure_steps=2 if TEST_MODE else 20, train_minutes=TRAIN_MINUTES,
              out_weights=f"{OUT}/adm_ft_ema.pt", out_log=f"{OUT}/train_log.json")
env = dict(os.environ, FT_CONFIG=json.dumps(CONFIG), NCCL_P2P_DISABLE="1", NCCL_IB_DISABLE="1", OMP_NUM_THREADS="2", USE_LIBUV="0",
           PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
t0 = time.time()
launch = [sys.executable, TRAIN_SCRIPT] if TEST_MODE else [sys.executable, "-m", "torch.distributed.run", "--standalone",
                                                          f"--nproc_per_node={len(DEVICES)}", TRAIN_SCRIPT]
subprocess.run(launch, env=env, check=True)
TIMES["training"] = time.time() - t0
TRAIN_LOG = pd.DataFrame(json.load(open(f"{OUT}/train_log.json")))
print(TRAIN_LOG.round(4).to_string())

## 3 · Base versus fine-tuned, on the same noisy images (fp16 inference, whole image reflect-padded to a multiple of 64)

In [ ]:
sys.path[:0] = [GD]
from guided_diffusion.script_util import create_model
from guided_diffusion import unet as gd_unet


def sdpa_attention(self, qkv):
    bs, width, length = qkv.shape
    ch = width // (3 * self.n_heads)
    q, k, v = qkv.reshape(bs * self.n_heads, ch * 3, length).split(ch, dim=1)
    out = F.scaled_dot_product_attention(q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2))
    return out.transpose(1, 2).reshape(bs, -1, length)


gd_unet.QKVAttentionLegacy.forward = sdpa_attention
FP16 = not TEST_MODE
BASE_STATE = None if TEST_MODE else torch.load(P["adm"], map_location="cpu", weights_only=True)
FT_STATE = {k: v.float() for k, v in torch.load(f"{OUT}/adm_ft_ema.pt", map_location="cpu", weights_only=True).items()}


def make_adm(device, state):
    torch.manual_seed(0)
    m = create_model(**ADM_FLAGS, use_fp16=FP16)
    if state is not None:
        m.load_state_dict(state)
    m.to(device).eval().requires_grad_(False)
    if FP16:
        m.convert_to_fp16()
    return m


import lpips

CTX = [dict(dev=d, base=make_adm(d, BASE_STATE), ft=make_adm(d, FT_STATE), lpips=lpips.LPIPS(net="alex", verbose=False).to(d)) for d in DEVICES]
del BASE_STATE, FT_STATE
BETAS = torch.linspace(1e-4, 0.02, 1000, dtype=torch.float64)
NEG_LOG_ABAR = (-torch.log(torch.cumprod(1 - BETAS, 0))).numpy()


def matched(sigma01):
    return float(np.interp(math.log1p(4 * sigma01 ** 2), NEG_LOG_ABAR, np.arange(1000))), 1 / (1 + 4 * sigma01 ** 2)


@torch.no_grad()
def tweedie(model, y, t, ab):
    """One-step posterior mean of y [B,3,H,W] at continuous step t (shared by the batch)."""
    h, w = y.shape[-2:]
    yp = F.pad(y, (0, (-w) % 64, 0, (-h) % 64), mode="reflect") if (h % 64 or w % 64) else y
    x = math.sqrt(ab) * (2 * yp - 1)
    eps = model(x, torch.full((x.shape[0],), t, device=y.device))[:, :3].float()
    return (((x - math.sqrt(1 - ab) * eps) / math.sqrt(ab) + 1) / 2)[..., :h, :w]


@torch.no_grad()
def two_steps(model, y, t, ab):
    """DMID's t -> 0 -> clean steps with eta = 0: deterministic, two evaluations."""
    h, w = y.shape[-2:]
    yp = F.pad(y, (0, (-w) % 64, 0, (-h) % 64), mode="reflect") if (h % 64 or w % 64) else y
    x = math.sqrt(ab) * (2 * yp - 1)
    e1 = model(x, torch.full((x.shape[0],), t, device=y.device))[:, :3].float()
    a0 = float(1 - BETAS[0])
    x_0 = math.sqrt(a0) * (x - math.sqrt(1 - ab) * e1) / math.sqrt(ab) + math.sqrt(1 - a0) * e1
    e2 = model(x_0, torch.zeros(x.shape[0], device=y.device))[:, :3].float()
    return (((x_0 - math.sqrt(1 - a0) * e2) / math.sqrt(a0) + 1) / 2)[..., :h, :w]


def sigma_hat_of(noisy):
    return float(estimate_sigma(noisy.permute(1, 2, 0).numpy().astype(np.float64), channel_axis=-1, average_sigmas=True))


M_GRID = [0.96, 0.98, 1.0, 1.02, 1.04]   # blind: start step t(m·σ̂), m chosen on the validation images
VAL_CASES = torch.load(VAL_FILE)
val_rows = []
for clean, s, noisy in VAL_CASES:
    sh = sigma_hat_of(noisy)
    for m in M_GRID:
        out = tweedie(CTX[0]["ft"], noisy[None].to(DEVICES[0]), *matched(m * sh))
        mse = ((out[0].clamp(0, 1) * 255).round() - clean.to(DEVICES[0]).float()).pow(2).mean().item()
        val_rows.append(dict(sigma=s, m=m, psnr=10 * math.log10(255 ** 2 / mse)))
M_HAT = float(pd.DataFrame(val_rows).groupby("m").psnr.mean().idxmax())
print("blind multiplier on σ̂ (validation):", M_HAT)
BLIND_SETS = {"CBSD68", "Kodak24", "McMaster"}   # blind and two-step variants: the three smaller sets only


_g = torch.exp(-torch.arange(-5, 6, dtype=torch.float64) ** 2 / (2 * 1.5 ** 2))
SSIM_WINDOW = (_g / _g.sum())[:, None] * (_g / _g.sum())[None, :]


def ssim(a, b):
    """skimage structural_similarity(gaussian_weights=True, sigma=1.5, use_sample_covariance=False, data_range=255)."""
    w = SSIM_WINDOW.to(a.device)[None, None].repeat(3, 1, 1, 1)
    conv = lambda x: F.conv2d(x[None], w, groups=3)[0]
    ux, uy = conv(a), conv(b)
    vx, vy, vxy = conv(a * a) - ux * ux, conv(b * b) - uy * uy, conv(a * b) - ux * uy
    c1, c2 = (0.01 * 255) ** 2, (0.03 * 255) ** 2
    return float((((2 * ux * uy + c1) * (2 * vxy + c2)) / ((ux * ux + uy * uy + c1) * (vx + vy + c2))).mean())


def scores(x, ref_u8, lp):
    out = (x[0].clamp(0, 1) * 255).round()
    a, b = out.double(), ref_u8.double()
    mse = (a - b).pow(2).mean().item()
    with torch.no_grad():
        lpv = lp(out.float()[None] / 127.5 - 1, ref_u8.float()[None] / 127.5 - 1).item()
    return dict(psnr=10 * math.log10(255 ** 2 / mse), ssim=ssim(a, b), lpips=lpv)


TEST = {}
with zipfile.ZipFile(P["datasets"]) as z:
    for ds in DATASETS:
        names = sorted([n for n in z.namelist() if n.startswith(f"Datasets/Gaussion/{ds}/") and not n.endswith("/")], key=natkey)
        names = names[:N_URBAN] if ds == "Urban100" else names
        TEST[ds] = [(n.split("/")[-1], np.asarray(Image.open(io.BytesIO(z.read(n))).convert("RGB"))) for n in names]
if TEST_MODE:
    TEST = {ds: [(n, im[:96, :128].copy()) for n, im in v[:2]] for ds, v in TEST.items()}


def noise_seed(ds, idx, sigma):
    return 1_000_000 * (DATASETS.index(ds) + 1) + 1000 * sigma + idx


def run_image(ctx, job):
    ds, idx, sigma = job
    name, clean_u8 = TEST[ds][idx]
    y = noisy_of(clean_u8, sigma, noise_seed(ds, idx, sigma))[None].to(ctx["dev"])
    ref = torch.from_numpy(clean_u8).permute(2, 0, 1).to(ctx["dev"])
    t, ab = matched(sigma / 255)
    outs = {"base ×1": tweedie(ctx["base"], y, t, ab), "fine-tuned ×1": tweedie(ctx["ft"], y, t, ab)}
    if EVAL_FLIP:
        flipped = tweedie(ctx["ft"], y.flip(-1), t, ab).flip(-1)
        outs["fine-tuned ×2"] = (outs["fine-tuned ×1"] + flipped) / 2
    if ds in BLIND_SETS:
        outs["fine-tuned two steps (η=0)"] = two_steps(ctx["ft"], y, t, ab)
        outs["fine-tuned blind ×1"] = tweedie(ctx["ft"], y, *matched(M_HAT * sigma_hat_of(y[0].cpu())))
    return [dict(dataset=ds, image=name, sigma=sigma, method=m, **scores(o, ref, ctx["lpips"])) for m, o in outs.items()]


def worker_pool(jobs, fn):
    """Run fn(ctx, job) over jobs with one thread per GPU; returns (results, errors)."""
    q, results, errors, lock = queue.Queue(), [], [], threading.Lock()
    for j in jobs:
        q.put(j)

    def loop(ctx):
        while time.time() - T_START < TIME_LIMIT_H * 3600:
            try:
                job = q.get_nowait()
            except queue.Empty:
                return
            try:
                out = fn(ctx, job)
                with lock:
                    results.append(out)
            except Exception:
                with lock:
                    errors.append((job, traceback.format_exc()))

    threads = [threading.Thread(target=loop, args=(c,)) for c in CTX]
    for th in threads:
        th.start()
    for th in threads:
        th.join()
    for job, tb in errors[:3]:
        print("ERROR", job, tb)
    return results, errors


t0 = time.time()
res, err = worker_pool([(ds, i, s) for ds in DATASETS for s in SIGMAS for i in range(len(TEST[ds]))], run_image)
SCORES = pd.DataFrame([r for rows in res for r in rows])
SCORES.to_csv(f"{OUT}/scores.csv", index=False)
TIMES["evaluation"] = time.time() - t0
print(f"{len(res)} images, {len(err)} errors, {TIMES['evaluation'] / 60:.1f} min")

## 4 · Results

In [ ]:
for metric in ("psnr", "ssim", "lpips"):
    print(f"\nmean {metric.upper()}")
    print(SCORES.pivot_table(index="method", columns=["dataset", "sigma"], values=metric).round(4 if metric != "psnr" else 3).to_string())
wide = SCORES.pivot_table(index=["dataset", "sigma", "image"], columns="method", values="psnr")
diff = (wide["fine-tuned ×1"] - wide["base ×1"]).groupby(["dataset", "sigma"])
print("\nfine-tuned ×1 − base ×1, PSNR (dB): mean, images better / n")
print(pd.DataFrame({"mean": diff.mean(), "better": diff.apply(lambda d: f"{int((d > 0).sum())}/{d.size}")}).to_string())
with open(f"{OUT}/summary.json", "w") as f:
    json.dump(dict(config={k: v for k, v in CONFIG.items() if k != "adm_flags"}, minutes={k: v / 60 for k, v in TIMES.items()},
                   blind_multiplier=M_HAT, init_weights=INIT,
                   train_log=TRAIN_LOG.to_dict("records"),
                   mean=SCORES.groupby(["dataset", "sigma", "method"])[["psnr", "ssim", "lpips"]].mean().reset_index().to_dict("records")),
              f, indent=1, default=float)
open(f"{OUT}/DONE", "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))
print(sorted(os.listdir(OUT)), f"total {(time.time() - T_START) / 60:.1f} min")